# Exploring LIBERO with FiftyOne

**Curate 100+ robot-manipulation demonstrations in minutes — multi-camera playback, CLIP embeddings, natural-language search, and outlier detection.**

[LIBERO](https://huggingface.co/datasets/physical-intelligence/libero) is a lifelong-learning benchmark of simulated robot manipulation demonstrations, organized into task suites (Spatial, Object, Goal, Long/90) with a language instruction attached to every episode. Each episode is recorded from two cameras: a third-person **agent view** and a **wrist** camera.

In this demo we will:

1. Download a small slice of LIBERO from the Hugging Face Hub (LeRobot format)
2. Import it into FiftyOne as a **grouped video dataset** — one group per episode, one slice per camera
3. Explore episodes, tasks, and synchronized camera views in the FiftyOne App
4. Extract representative frames and compute **CLIP embeddings**
5. Visualize the dataset in **UMAP embedding space** (task suites cluster visually!)
6. Run **natural-language similarity search** ("a robot arm reaching for a bowl")
7. Use **uniqueness** to surface anomalous / near-duplicate demonstrations and tag episodes for review

> ⚠️ Demonstration quality silently determines imitation-learning quality. The punchline of this demo is: *find the weird demos before you train on them.*

## 0. Environment setup (macOS)

Run these commands in **Terminal** (not in this notebook) to create an isolated environment for this demo. Requires **Python 3.10–3.12** (`python3 --version`; on newer interpreters, install a supported one, e.g. `brew install python@3.12`, and use `python3.12` below).

This demo pins **`fiftyone==1.19.0`**. FiftyOne 1.20.0 has a known bug that crashes the App's Embeddings panel on open (a duplicate MUI dependency is bundled by `@fiftyone/embeddings-v2`); see voxel51/fiftyone#8181. Use 1.19.0 until a patched release (1.20.1+) is available.

```bash
# -- 1. System dependency: ffmpeg with AV1 decode support (LeRobot videos are often AV1)
brew install ffmpeg
ffmpeg -decoders | grep av1        # should list libdav1d / libaom-av1 / av1

# -- 2. Create a dedicated project folder + virtual environment
mkdir -p libero-fiftyone-demo && cd libero-fiftyone-demo
python3 -m venv .venv-libero-demo
source .venv-libero-demo/bin/activate

# -- 3. Install pinned FiftyOne + demo dependencies
pip install --upgrade pip
pip install "fiftyone==1.19.0"
pip install "huggingface_hub[hf_transfer]" ffmpeg-python pyarrow \
            torch torchvision umap-learn matplotlib \
            jupyterlab ipykernel ipywidgets

# -- 4. Clone the community LeRobot v3.0 importer for FiftyOne
git clone https://github.com/harpreetsahota204/fiftyone_lerobot_importer.git

# -- 5. Register the venv as a Jupyter kernel and launch
python -m ipykernel install --user --name libero-fiftyone-demo \
       --display-name "Python (libero-fiftyone-demo)"
jupyter lab
```

Then open this notebook and select the **Python (libero-fiftyone-demo)** kernel.

**Notes**

- FiftyOne ships its own embedded MongoDB — no database setup needed on macOS.
- If you use conda, its bundled `ffmpeg` often lacks AV1 support; prefer the Homebrew one (`conda remove --force ffmpeg`).
- On Apple Silicon, PyTorch will use MPS automatically where possible; CPU is also fine at this demo's scale.

## 1. Sanity checks

Verify the environment before downloading anything.

In [ ]:
import shutil
import subprocess
import sys

import fiftyone as fo
import fiftyone.brain as fob

print(f"Python:   {sys.version.split()[0]}")
print(f"FiftyOne: {fo.__version__}")

assert fo.__version__ == "1.19.0", (
    f"This demo pins fiftyone==1.19.0 (1.20.0 crashes the Embeddings panel; "
    f"see voxel51/fiftyone#8181). Found {fo.__version__}."
)

assert shutil.which("ffmpeg"), "ffmpeg not found — run `brew install ffmpeg`"
decoders = subprocess.run(
    ["ffmpeg", "-decoders"], capture_output=True, text=True
).stdout
print("AV1 decode support:", "✅" if "av1" in decoders else "❌ (brew install ffmpeg)")


In [ ]:
import sys
from pathlib import Path

# Point Python at the cloned importer repo (adjust if you cloned elsewhere)
IMPORTER_DIR = Path.home() / "libero-fiftyone-demo" / "fiftyone_lerobot_importer"
assert IMPORTER_DIR.exists(), f"Importer repo not found at {IMPORTER_DIR}"
sys.path.insert(0, str(IMPORTER_DIR))

from lerobot_importer import LeRobotDataset, apply_lerobot_field_descriptions
print("LeRobot importer loaded ✅")


## 2. Download a slice of LIBERO

We use **`lerobot/libero`** — the official LeRobot **v3.0** release of LIBERO (the `physical-intelligence/libero` snapshot is legacy v2.0, which the importer can't read). The full dataset is large, so we grab the metadata first (tiny), verify the format version, then pull only the **first data/video chunk** — enough episodes for a compelling demo.

In [ ]:
import json
from huggingface_hub import snapshot_download

# Note: physical-intelligence/libero is in legacy v2.0 format. lerobot/libero is
# the official v3.0 re-release of the same 1,693 episodes — use that one.
REPO_ID = "lerobot/libero"
DATA_DIR = Path.home() / "libero-fiftyone-demo" / "libero_data"

# Metadata only — a few MB
snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=DATA_DIR,
    allow_patterns=["meta/**", "README.md"],
)

info = json.loads((DATA_DIR / "meta" / "info.json").read_text())
version = str(info.get("codebase_version", "unknown"))
print(f"LeRobot codebase_version: {version}")
print(f"Robot type:               {info.get('robot_type')}")
print(f"Total episodes:           {info.get('total_episodes')}")
print(f"FPS:                      {info.get('fps')}")
print(f"Camera features:          {[k for k in info.get('features', {}) if 'image' in k]}")

if not version.startswith("v3") and not version.startswith("3"):
    print(
        "\n⚠️  This snapshot is not in LeRobot v3.0 format, which the importer requires."
        "\nEasiest fix: use a v3.0 release of LIBERO instead of converting —"
        "\n    REPO_ID = 'lerobot/libero'            # all 4 suites, 1,693 episodes"
        "\n    REPO_ID = 'lerobot/libero_object_image'  # or a single suite"
        "\nIf you must convert a custom v2.x dataset:"
        "\n    pip install lerobot"
        "\n    python -m lerobot.datasets.v30.convert_dataset_v21_to_v30 --repo-id=<your/dataset>"
        "\n(v2.0 datasets need the v20→v21 converter first.)"
    )


In [ ]:
# Media: first chunk only (covers both v3.0 and legacy directory layouts)
snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=DATA_DIR,
    allow_patterns=[
        "meta/**",
        "data/chunk-000/**",
        "videos/chunk-000/**",      # v3.0 layout
        "videos/**/chunk-000/**",   # per-camera layout
    ],
)

n_files = sum(1 for _ in DATA_DIR.rglob("*") if _.is_file())
size_gb = sum(f.stat().st_size for f in DATA_DIR.rglob("*") if f.is_file()) / 1e9
print(f"Downloaded {n_files} files ({size_gb:.2f} GB) to {DATA_DIR}")


## 3. Import into FiftyOne

The importer creates a **grouped video dataset**: each group is one episode; each slice is one camera view. Videos are re-encoded to H.264 automatically so they play in the App. We cap the import at 20 episodes to keep things snappy — raise `MAX_EPISODES` if you want more.

In [ ]:
MAX_EPISODES = 20
DATASET_NAME = "libero-demo"

if fo.dataset_exists(DATASET_NAME):
    fo.delete_dataset(DATASET_NAME)

dataset = fo.Dataset.from_dir(
    dataset_dir=str(DATA_DIR),
    dataset_type=LeRobotDataset,
    name=DATASET_NAME,
    max_samples=MAX_EPISODES,
)
apply_lerobot_field_descriptions(dataset)
dataset.persistent = True

print(dataset)


In [ ]:
print("Camera slices:", dataset.group_slices)
print("Default slice:", dataset.default_group_slice)
print()
print("Tasks in this slice of LIBERO:")
for task, count in dataset.count_values("task").items():
    print(f"  {count:>3} × {task}")


In [ ]:
# Peek at one episode: multi-camera group + per-frame robot state
group = next(iter(dataset.iter_groups()))
for slice_name, sample in group.items():
    print(f"[{slice_name}] episode {sample.episode_index}: {sample.task!r}")

sample = dataset.first()
frame = sample.frames[1]
print("\nFrame 1 state:  ", frame["observation_state"][:4], "…")
print("Frame 1 action: ", frame["action"][:4], "…")


## 4. Prepare videos for browser playback

The importer transcodes LIBERO's AV1 source clips to H.264, but the resulting files carry an **H.264 level of 1.2** — a value only valid for tiny (sub-QCIF) frames. Our clips are 256×256, so that header is out of spec, and browsers handle the mismatch inconsistently: you'll see thumbnails in the grid (extracted server-side by ffmpeg) but a **black or missing video in the expanded modal**, often differing between Chrome and Safari.

The fix is a one-time re-encode that forces a browser-safe profile/level/pixel-format. We shell out to `ffmpeg` directly (Main profile, level 3.1, `yuv420p`) rather than using FiftyOne's re-encode helper, which reproduces the bad level. This is fast at 256×256.

> If your clips already play in the App modal, you can skip to section 5 — but running this is harmless and idempotent (`_web.mp4` files are only created once).

In [ ]:
import subprocess
from pathlib import Path

video_view = dataset.select_group_slices(media_type="video")

def to_web_mp4(src: str) -> str:
    """Re-encode one clip to browser-safe H.264 (Main / level 3.1 / yuv420p)."""
    src = Path(src)
    dst = src.with_name(src.stem + "_web.mp4")
    if not dst.exists():
        subprocess.run(
            [
                "ffmpeg", "-y", "-loglevel", "error", "-i", str(src),
                "-c:v", "libx264", "-profile:v", "main", "-level", "3.1",
                "-pix_fmt", "yuv420p", "-movflags", "+faststart", "-an",
                str(dst),
            ],
            check=True,
        )
    return str(dst)

# Re-encode every camera clip and repoint the samples at the web-safe copies
n = 0
for sample in video_view.iter_samples(progress=True, autosave=True):
    if "_web.mp4" not in sample.filepath:
        sample.filepath = to_web_mp4(sample.filepath)
        n += 1
print(f"Re-encoded / repointed {n} clips")

dataset.select_group_slices(media_type="video").compute_metadata(overwrite=True)


In [ ]:
# Sanity check: level should now read 31 (i.e. 3.1), not 12
import subprocess

probe_path = dataset.select_group_slices(media_type="video").first().filepath
out = subprocess.run(
    ["ffprobe", "-v", "error", "-select_streams", "v:0",
     "-show_entries", "stream=profile,pix_fmt,level",
     "-of", "default=nw=1", probe_path],
    capture_output=True, text=True,
).stdout
print(out or "ffprobe returned nothing — check the path")


## 5. Launch the FiftyOne App

Things to try in the App:

- Use the **group slices dropdown** (top of the grid) to flip between agent view and wrist camera — playback stays synchronized per episode
- Filter by `task` in the sidebar to see all demonstrations of a single instruction
- Click any episode to scrub through the video with its per-frame `observation_state` and `action` fields

In [ ]:
session = fo.launch_app(dataset, port=5151, auto=False)
session.show()


## 6. Extract representative frames

Embedding models operate on images, so we sample the agent-view videos at 1 fps into a companion **frames dataset**, carrying each episode's task label along.

In [ ]:
# Prefer the third-person camera for embeddings (wrist views are extreme close-ups)
agent_slice = next(
    (s for s in dataset.group_slices if "wrist" not in s.lower()),
    dataset.group_slices[0],
)
print(f"Using camera slice: {agent_slice!r}")

videos = dataset.select_group_slices(agent_slice)

frames_view = videos.to_frames(sample_frames=True, fps=1)

FRAMES_NAME = "libero-demo-frames"
if fo.dataset_exists(FRAMES_NAME):
    fo.delete_dataset(FRAMES_NAME)
frames = frames_view.clone(FRAMES_NAME)
frames.persistent = True

# Propagate task + episode from parent videos onto the frame images
task_map = {s.id: (s.task, s.episode_index) for s in videos}
parent_ids = frames.values("sample_id")
frames.set_values("task", [task_map[i][0] for i in parent_ids])
frames.set_values("episode_index", [task_map[i][1] for i in parent_ids])

print(f"{len(frames)} frames extracted from {len(videos)} episodes")


## 7. CLIP embeddings + UMAP visualization

One call computes CLIP embeddings for every frame, reduces them with UMAP, and stores everything on the dataset. Coloring by `task`, demonstrations of the same instruction cluster together — and points that sit far from their cluster are exactly the frames worth inspecting.

In [ ]:
results = fob.compute_visualization(
    frames,
    model="clip-vit-base32-torch",   # downloads weights on first run
    embeddings="clip_emb",           # embeddings stored on each sample
    brain_key="clip_umap",
    method="umap",
    seed=51,
)


### Static plot (always works)

Before the interactive options, here's the dependable one: a static matplotlib scatter of the UMAP points colored by task. It renders inline in any kernel state — no App, no widgets, no version-matching — and doubles as a figure you can drop straight into a slide or blog post. The story is already visible here: same-task demonstrations land together, and stray points are candidate outliers.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

pts = np.array(results.points)          # (n_frames, 2) UMAP coordinates
tasks = frames.values("task")
uniq = sorted(set(tasks))
color_idx = {t: i for i, t in enumerate(uniq)}

plt.figure(figsize=(10, 8))
plt.scatter(
    pts[:, 0], pts[:, 1],
    c=[color_idx[t] for t in tasks],
    cmap="tab20", s=12, alpha=0.85,
)
plt.title("CLIP embeddings of LIBERO frames (UMAP), colored by task")
plt.xlabel("UMAP-1"); plt.ylabel("UMAP-2")
plt.tight_layout()
plt.savefig("libero_umap.png", dpi=150)
plt.show()
print(f"{len(uniq)} tasks, {len(pts)} frames")


### Interactive: the App's Embeddings panel

For live **lasso-a-cluster → filter-the-grid** interactivity, use the **Embeddings panel** inside the FiftyOne App. It renders in the browser (no Jupyter-widget plumbing), and must be pointed at the *frames* dataset — the brain runs live there, not on the video dataset.

1. Run the cell below to point the App at the frames dataset
2. Refresh your browser tab, then click the **`+`** next to the *Samples* tab → **Embeddings**
3. Select brain key **`clip_umap`**, then color by **`task`**
4. Lasso any cluster — the grid updates to just those frames

In [ ]:
# Point the App at the frames dataset (the Embeddings panel reads its brain runs).
assert "clip_umap" in frames.list_brain_runs(), (
    "clip_umap not found on the frames dataset — run the compute cell above first"
)

# Clear any attached plots as a safety measure, then switch the App's dataset.
session.plots.clear()
session.dataset = frames
# No session.show() — just refresh your existing localhost:5151 browser tab.
print("App now on:", session.dataset.name)


> ♻️ **Resuming after a kernel restart?** Everything is persisted — reload and continue:
> ```python
> dataset = fo.load_dataset("libero-demo")
> frames  = fo.load_dataset("libero-demo-frames")
> results = frames.load_brain_results("clip_umap")
> session = fo.launch_app(frames, auto=False)
> ```

## 8. Natural-language search

CLIP is a multimodal model, so the same embeddings power free-text queries. Build a similarity index (it reuses the stored embeddings — nothing is recomputed) and search the demonstrations in plain English.

In [ ]:
fob.compute_similarity(
    frames,
    model="clip-vit-base32-torch",
    embeddings="clip_emb",
    brain_key="clip_sim",
)


In [ ]:
QUERY = "a robot arm reaching toward a bowl on a stove"

hits = frames.sort_by_similarity(QUERY, k=25, brain_key="clip_sim")
session.view = hits
print(f"Top episodes for {QUERY!r}:")
for task, count in hits.count_values("task").items():
    print(f"  {count:>2} frames — {task}")


Try your own queries: `"gripper holding a wine bottle"`, `"open drawer"`, `"cluttered tabletop"` — then check the App to verify the hits visually.

## 9. Find anomalies and near-duplicates

`compute_uniqueness` scores every frame by how distinct it is within embedding space:

- **High uniqueness** → visual outliers: odd arm configurations, dropped objects, rendering glitches — candidate *bad demos*
- **Low uniqueness** → near-duplicates: heavily redundant frames/episodes that add little training signal

In [ ]:
fob.compute_uniqueness(frames, embeddings="clip_emb")

print("Most anomalous frames:")
session.view = frames.sort_by("uniqueness", reverse=True)


In [ ]:
# Roll frame-level findings back up to the episode level:
# tag every episode that produced a top-5% outlier frame for human review
n_outliers = max(1, int(0.05 * len(frames)))
outlier_frames = frames.sort_by("uniqueness", reverse=True).limit(n_outliers)
flagged_episode_ids = set(outlier_frames.values("sample_id"))

review = dataset.select_group_slices(agent_slice).select(list(flagged_episode_ids))
review.tag_samples("review")

print(f"Tagged {len(flagged_episode_ids)} episodes for review:")
for task, count in review.count_values("task").items():
    print(f"  {count} × {task}")

# Back in the App: filter by the 'review' tag in the sidebar to audit them
session.view = review


## 10. Save named views for the highlights

Saved views persist a named sequence of view stages on a dataset, so anyone who opens it later can jump straight to a highlight from the App's **view dropdown** (top-left) — no code required. We bookmark the moments worth showing in a walkthrough.

Note *where* each view lives: the per-task and anomaly views are over the **frames** dataset (that's where the embeddings and `uniqueness` scores are), while the review view is over the **video** dataset (that's what you actually scrub). Both datasets show their own saved views in the App.

In [ ]:
from fiftyone import ViewField as F

# --- Views on the FRAMES dataset (embeddings live here) ------------------

# 1) One representative task, as a filtered grid you can eyeball
example_task = frames.distinct("task")[0]
frames.save_view(
    "task: " + example_task,
    frames.match(F("task") == example_task),
    description="All sampled frames for a single LIBERO instruction",
)

# 2) The most anomalous frames — candidate bad demos
n_outliers = max(1, int(0.05 * len(frames)))
frames.save_view(
    "anomalies (top 5% uniqueness)",
    frames.sort_by("uniqueness", reverse=True).limit(n_outliers),
    description="Highest-uniqueness frames: odd poses, dropped objects, glitches",
)

# 3) A saved text-similarity query (edit the phrase, re-open anytime)
QUERY = "a robot arm reaching toward a bowl on a stove"
frames.save_view(
    "search: bowl on stove",
    frames.sort_by_similarity(QUERY, k=25, brain_key="clip_sim"),
    description=f"Top matches for: {QUERY!r}",
)

# --- View on the VIDEO dataset (scrubbable episodes) ---------------------

# 4) Episodes flagged for human review (tagged in section 9)
dataset.save_view(
    "review: flagged episodes",
    dataset.select_group_slices(media_type="video").match_tags("review"),
    description="Episodes that produced a top-5% outlier frame",
)

print("Saved views on frames dataset:", frames.list_saved_views())
print("Saved views on video dataset: ", dataset.list_saved_views())


**Using them in the App:** load either dataset (`session.dataset = frames` or `session.dataset = dataset`), then pick a view from the dropdown at the top-left of the App. To load one programmatically:

```python
session.dataset = frames
session.view = frames.load_saved_view("anomalies (top 5% uniqueness)")
```

Saved views are lightweight — they store the *stages*, not copies of the data — and they travel with the dataset, so they persist across kernel restarts and are there for anyone who opens `libero-demo` later.

## 11. Where to take it next

- **Scale up** — raise `MAX_EPISODES` or download more chunks; the workflow is identical at 6,500 episodes
- **Compare cameras** — repeat the embedding analysis on the wrist slice and diff the clusters
- **Model in the loop** — run a VLM over frames and flag episodes where the caption disagrees with the `task` instruction, or log a trained policy's predicted actions per frame and sort episodes by divergence from the demonstrations
- **Export** — `review.export(...)` or write flagged episode indices back out to filter your LeRobot training config

Cleanup (optional):

```python
fo.delete_dataset("libero-demo")
fo.delete_dataset("libero-demo-frames")
```
